In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
!pip install librosa pandas tqdm

In [16]:
import os
import librosa
import pandas as pd
from tqdm import tqdm
from tqdm import tqdm
import numpy as np

In [6]:
DATASET_PATH = "/content/drive/MyDrive/speech_data"
os.listdir(DATASET_PATH)
os.listdir("/content/drive/MyDrive/speech_data/demented")

['Viv Nicholson',
 'Tony Parkes',
 'Vampiro',
 'Woody Durham',
 'Trevor Peacock',
 'Tony Bennett',
 'Tim Conway',
 'Unita Blackwell',
 'Terry Pratchett',
 'Thomas Dorsey',
 'Ronald Reagan',
 'Terry Jones',
 'Teresa Gorman',
 'Ross Mac Donald',
 'Ted Turner',
 'Stan Bowles',
 'Stella Stevens',
 'Steve Lawrence',
 'Ronan O_Rahilly',
 'Sparky Anderson',
 'Peter Reveen',
 'Richard Gwyn',
 'Ralph Klein',
 'Robert Anderson',
 'Ray Galton',
 'Peter Max',
 'Peter Falk',
 'Robin Williams',
 'Ray Dolby',
 'Pauline Phillips',
 'Paul Hornung',
 'Pat Pariseau',
 'Omar Sharif',
 'Leon Redbone',
 'Maurice Hinchey',
 'Lerone Bennett Jr',
 'John Mackey',
 'Otis Chandler',
 'Jonathan Miller',
 'Maureen Forrester',
 'Joe Conley',
 'Jesse Helms',
 'Jimmy Fratianno',
 'Jimmy Calderwood',
 'Jessejackson',
 'Jim McLean',
 'Jim Neidhart',
 'Jeanne Little',
 'James Stewart',
 'Jack Webster',
 'Jack Hanna',
 'Ian McCaskill',
 'Iris Murdoch',
 'Gale Sayers',
 'George Robb',
 'Ian Holm',
 'George Klein',
 'Greg F

In [7]:
!pip install transformers
!pip install torch
!pip install librosa

In [8]:
import torch
import librosa

from transformers import Wav2Vec2Processor
from transformers import Wav2Vec2Model

In [9]:
processor = Wav2Vec2Processor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [10]:
import os
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/speech_data"

rows = []

for label_name in ["demented", "non-demented"]:

    label = 1 if label_name == "demented" else 0

    class_path = os.path.join(DATASET_PATH, label_name)

    for speaker in os.listdir(class_path):

        speaker_path = os.path.join(class_path, speaker)

        for file in os.listdir(speaker_path):

            if file.lower().endswith(".wav"):

                file_path = os.path.join(speaker_path, file)

                rows.append({
                    "file_path": file_path,
                    "speaker": speaker,
                    "label": label
                })

df = pd.DataFrame(rows)

print("Total samples:", len(df))
df.head()

Total samples: 353


,file_path,speaker,label
0,/content/drive/MyDrive/speech_data/demented/Vi...,Viv Nicholson,1
1,/content/drive/MyDrive/speech_data/demented/To...,Tony Parkes,1
2,/content/drive/MyDrive/speech_data/demented/To...,Tony Parkes,1
3,/content/drive/MyDrive/speech_data/demented/Va...,Vampiro,1
4,/content/drive/MyDrive/speech_data/demented/Va...,Vampiro,1


In [11]:
from sklearn.model_selection import train_test_split

speakers = df["speaker"].unique()

train_spk, test_spk = train_test_split(
    speakers,
    test_size=0.2,
    random_state=42
)

train_df = df[df["speaker"].isin(train_spk)]
test_df = df[df["speaker"].isin(test_spk)]

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

Train samples: 284
Test samples: 69


In [28]:
##FEATURE EXTRACTION
import numpy as np
import librosa
import torch

def extract_features(file_path):

    audio, sr = librosa.load(file_path, sr=16000)

    # -------------------
    # Wav2Vec Embeddings
    # -------------------

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state

    wav2vec_features = embeddings.mean(dim=1).squeeze().cpu().numpy()

    # -------------------
    # Pause Features
    # -------------------

    intervals = librosa.effects.split(audio, top_db=25)

    pauses = []

    prev_end = 0

    for start, end in intervals:

        pause = start - prev_end

        if pause > 0:
            pauses.append(pause / sr)

        prev_end = end

    if len(pauses) == 0:
        pause_features = [0, 0, 0]
    else:
        pause_features = [
            np.mean(pauses),
            np.max(pauses),
            len(pauses)
        ]

    # -------------------
    # Energy Feature
    # -------------------

    rms = librosa.feature.rms(y=audio)
    energy = np.mean(rms)

    # -------------------
    # Combine Features
    # -------------------

    final_features = np.concatenate([
        wav2vec_features,
        pause_features,
        [energy]
    ])

    return final_features

In [29]:
X = []
y = []

from tqdm import tqdm

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):

    features = extract_features(row["file_path"])

    X.append(features)
    y.append(row["label"])

X = np.array(X)
y = np.array(y)

print(X.shape)

100%|██████████| 284/284 [02:39<00:00,  1.78it/s]

(284, 772)


In [30]:
X_test = []
y_test = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    features = extract_features(row["file_path"])

    X_test.append(features)
    y_test.append(row["label"])

X_test = np.array(X_test)
y_test = np.array(y_test)

100%|██████████| 69/69 [00:41<00:00,  1.67it/s]


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

In [38]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)

X = pca.fit_transform(X)
X_test = pca.transform(X_test)

In [39]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=4000,
    class_weight="balanced",
    C=0.5
)

clf.fit(X, y)

LogisticRegression(C=0.5, class_weight='balanced', max_iter=4000)

In [40]:
from sklearn.metrics import classification_report

preds = clf.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.68      0.57      0.62        44
           1       0.41      0.52      0.46        25

    accuracy                           0.55        69
   macro avg       0.54      0.54      0.54        69
weighted avg       0.58      0.55      0.56        69



In [41]:
import joblib

joblib.dump(clf, "dementia_model.pkl")

['dementia_model.pkl']

In [42]:
joblib.dump(pca, "pca.pkl")

['pca.pkl']

In [43]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [44]:
from google.colab import files

files.download("dementia_model.pkl")
files.download("scaler.pkl")
files.download("pca.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>